# 01 - Prompt Evaluation

In [1]:
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from app.models.schemas import (
    CalculatorInput, BusinessSector, EmployeeCount, RepeatPeriod
)
from app.services.calculator import calculate
from app.services.ai_analysis import analyze
from app.prompts.analysis import build_analysis_prompt

## Test cases
We highlight real-world examples from various industries.

In [2]:
test_cases = [
    {
        "name": "Счетоводство - въвеждане на фактури",
        "input": CalculatorInput(
            sector=BusinessSector.finance,
            employee_count=EmployeeCount.medium,
            process_description="Ръчно въвеждаме фактури от PDF в счетоводния софтуер. Получаваме ги по имейл и ги преписваме една по една.",
            repeat_count=20,
            repeat_period=RepeatPeriod.daily,
            people_involved=2,
            minutes_per_cycle=8,
            hourly_rate_eur=12.0,
        )
    },
    {
        "name": "Търговия - обработка на поръчки",
        "input": CalculatorInput(
            sector=BusinessSector.trade,
            employee_count=EmployeeCount.small,
            process_description="Получаваме поръчки по телефон и Viber, записваме ги в Excel и после ги прехвърляме в системата за склад ръчно.",
            repeat_count=15,
            repeat_period=RepeatPeriod.daily,
            people_involved=1,
            minutes_per_cycle=10,
            hourly_rate_eur=8.0,
        )
    },
    {
        "name": "Услуги - изпращане на оферти",
        "input": CalculatorInput(
            sector=BusinessSector.services,
            employee_count=EmployeeCount.small,
            process_description="След всяка консултация с клиент изготвяме оферта в Word, копираме данните от CRM-а и я изпращаме по имейл.",
            repeat_count=5,
            repeat_period=RepeatPeriod.daily,
            people_involved=1,
            minutes_per_cycle=20,
            hourly_rate_eur=15.0,
        )
    },
    {
        "name": "Логистика - справки за доставки",
        "input": CalculatorInput(
            sector=BusinessSector.logistics,
            employee_count=EmployeeCount.large,
            process_description="Всяка сутрин събираме статусите на доставките от три различни куриерски системи и правим обобщен отчет в Excel за мениджмънта.",
            repeat_count=1,
            repeat_period=RepeatPeriod.daily,
            people_involved=3,
            minutes_per_cycle=45,
            hourly_rate_eur=10.0,
        )
    },
]

## Prompt Overview
We can see exactly what we're sending to OpenAI.

In [3]:
for case in test_cases:
    result = calculate(case['input'])
    prompt = build_analysis_prompt(case['input'], result)
    print(f"{'='*60}")
    print(f"СЛУЧАЙ: {case['name']}")
    print(f"{'='*60}")
    print(prompt)
    print()

СЛУЧАЙ: Счетоводство - въвеждане на фактури
Ти си експерт по бизнес автоматизация и изкуствен интелект. Анализирай следния бизнес процес и върни отговор САМО като валиден JSON без никакъв друг текст.

КОНТЕКСТ ЗА БИЗНЕСА:
- Сфера: Финанси/Счетоводство
- Размер: 6-20 служители

ПРОЦЕСЪТ:
- Описание: Ръчно въвеждаме фактури от PDF в счетоводния софтуер. Получаваме ги по имейл и ги преписваме една по една.
- Повторения: 20 пъти на ден
- Участници: 2 човека
- Времетраене: 8 минути на цикъл

ИЗЧИСЛЕНИ ЗАГУБИ:
- Изгубени часове/месец: 117.33
- Изгубена сума/месец: 1407.96€
- Изгубена сума/година: 16895.52€

ПРИМЕРИ ЗА КАЧЕСТВЕН АНАЛИЗ:
Пример 1:
{
  "process_category": "Обработка на фактури",
  "automation_complexity": "Лесна",
  "analysis_text": "Процесът е класически случай на ръчно въвеждане на структурирани данни — задача, която AI решава надеждно. OCR модел извлича данните от PDF фактурите, след което RPA робот ги въвежда директно в счетоводния софтуер без човешка намеса. При 20 фактури

## Calculations
We double-check the math just to be sure.

In [4]:
for case in test_cases:
    result = calculate(case['input'])
    print(f"{'='*60}")
    print(f"СЛУЧАЙ: {case['name']}")
    print(f"  Часове/месец: {result.hours_lost_per_month}h")
    print(f"  Сума/месец: {result.cost_per_month_eur}€")
    print(f"  Сума/година: {result.cost_per_year_eur}€")
    print()

СЛУЧАЙ: Счетоводство - въвеждане на фактури
  Часове/месец: 117.33h
  Сума/месец: 1407.96€
  Сума/година: 16895.52€

СЛУЧАЙ: Търговия - обработка на поръчки
  Часове/месец: 55.0h
  Сума/месец: 440.0€
  Сума/година: 5280.0€

СЛУЧАЙ: Услуги - изпращане на оферти
  Часове/месец: 36.67h
  Сума/месец: 550.05€
  Сума/година: 6600.6€

СЛУЧАЙ: Логистика - справки за доставки
  Часове/месец: 49.5h
  Сума/месец: 495.0€
  Сума/година: 5940.0€



## AI Analysis
We send the data to OpenAI and evaluate the quality of the responses.

In [5]:
for case in test_cases:
    result = calculate(case['input'])
    ai = analyze(case['input'], result)
    print(f"{'='*60}")
    print(f"СЛУЧАЙ: {case['name']}")
    print(f"  Категория: {ai.process_category}")
    print(f"  Сложност: {ai.automation_complexity}")
    print(f"  Анализ: {ai.analysis_text}")
    print()

СЛУЧАЙ: Счетоводство - въвеждане на фактури
  Категория: Въвеждане на фактури
  Сложност: Лесна
  Анализ: Процесът на ръчно въвеждане на фактури от PDF в счетоводния софтуер е идеален за автоматизация. С помощта на OCR технологии, данните могат да бъдат извлечени автоматично, а RPA робот може да ги въведе директно в системата. Това би спестило 117.33 часа месечно на двамата служители, което се равнява на 1407.96€ загуба месечно.

СЛУЧАЙ: Търговия - обработка на поръчки
  Категория: Обработка на поръчки
  Сложност: Лесна
  Анализ: Процесът на ръчно записване на поръчки от телефон и Viber в Excel, последвано от прехвърляне в складовата система, е типичен случай за автоматизация. С помощта на RPA робот, данните могат да се извлекат автоматично и да се въведат директно в системата, спестявайки 55 часа месечно и 440.0€ от загуби. Това ще позволи на служителя да се фокусира върху по-стойностни задачи.

СЛУЧАЙ: Услуги - изпращане на оферти
  Категория: Генериране на оферти
  Сложност: Лесна
 